# 14.5 怎麼讓特徵控制另一組特徵？

# 第 14 章：逐項理解現代文字模型的設計

前置：第4章基準。這章像替熟悉的車逐一換零件，每節可獨立進入；不要同時換六項再猜哪項有效。結構／數值可先CPU驗證，品質與成本需匹配預算後訓練。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：成對feature像鐘面指針：用相對旋轉記位置**

單一pair的旋轉示意；不同pair有不同角速度。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rope.svg")))

**先想一想：**gate為負時，內容可能反號嗎？

想像一路產生內容，另一路調音量。gate逐feature控制內容，有些訊號可以保留、有些抑制。多一條Linear也增加參數。

**把直覺寫成數學：**SwiGLU=SiLU(W_gate x)⊙(W_up x)，再經W_down；gate不是0–1機率。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
up = torch.tensor([2.0, 2.0, 2.0])
gate = torch.tensor([-2.0, 0.0, 2.0])
print("SiLU gate", F.silu(gate), "調節後", up * F.silu(gate))
from tiny_perceptron.modern import DenseFFN

for act in ("gelu", "swiglu"):
    layer = DenseFFN(8, hidden=16, activation=act)
    print(act, sum(p.numel() for p in layer.parameters()))

**如何讀結果：**公平比較可縮SwiGLU hidden以匹配參數；不能只說相同hidden就是相同預算。

**只改一件事：**只把gate改成全0。
